<a href="https://colab.research.google.com/github/dakshraj00/deep-fake-detection/blob/x_automation/resnet01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("hi")

hi


In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/Final_Dataset'


dataset_path = '/content/drive/Shareddrives/Final_Dataset'

dataset_path = '/content/drive/MyDrive/../Shared with me/Final_Dataset'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

dataset_path = '/content/drive/MyDrive/Final_Dataset'

print("Main folder contents:", os.listdir(dataset_path))
print("Total Fake images:", len(os.listdir(f"{dataset_path}/Fake")))
print("Total Real images:", len(os.listdir(f"{dataset_path}/Real")))

Main folder contents: ['dataset.csv', 'Real', 'Fake']
Total Fake images: 7013
Total Real images: 5890


In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_path}/dataset.csv")
print(df.shape)
print(df.head())
print(df['label'].value_counts())

(12890, 2)
                                                path label
0  /kaggle/input/stylegan-and-stylegan2-combined-...  Real
1  /kaggle/input/stylegan-and-stylegan2-combined-...  Real
2  /kaggle/input/stylegan-and-stylegan2-combined-...  Fake
3  /kaggle/input/stylegan-and-stylegan2-combined-...  Real
4  /kaggle/input/stylegan-and-stylegan2-combined-...  Real
label
Fake    7000
Real    5890
Name: count, dtype: int64


In [ ]:
print(df.columns)
print(df.head(2))

Index(['path', 'label'], dtype='object')
                                                path label
0  /kaggle/input/stylegan-and-stylegan2-combined-...  Real
1  /kaggle/input/stylegan-and-stylegan2-combined-...  Real


In [ ]:
import shutil
import os
from tqdm.notebook import tqdm

# Check if the directory exists before attempting to remove it
if os.path.exists('/content/Final_Dataset'):
    print("🗑️ Removing incomplete copy...")
    shutil.rmtree('/content/Final_Dataset')
    print("✅ Removed!")

src = '/content/drive/MyDrive/Final_Dataset'
dst = '/content/Final_Dataset'
os.makedirs(dst, exist_ok=True)

shutil.copy2(os.path.join(src, 'dataset.csv'), os.path.join(dst, 'dataset.csv'))
print("📄 CSV copied!")

for folder in ['Real', 'Fake']:
    src_folder = os.path.join(src, folder)
    dst_folder = os.path.join(dst, folder)
    os.makedirs(dst_folder, exist_ok=True)

    files = os.listdir(src_folder)
    print(f"\n📁 Copying {folder} ({len(files)} files)...")

    for file in tqdm(files, desc=folder, unit="img"):
        shutil.copy2(os.path.join(src_folder, file), os.path.join(dst_folder, file))

    print(f"✅ {folder} done!")

print("\n🎉 Dataset ready!")

📄 CSV copied!

📁 Copying Real (5890 files)...


Real:   0%|          | 0/5890 [00:00<?, ?img/s]

✅ Real done!

📁 Copying Fake (7013 files)...


Fake:   0%|          | 0/7013 [00:00<?, ?img/s]

✅ Fake done!

🎉 Dataset ready!


In [ ]:
df['path'] = df['path'].str.replace(
    '/kaggle/input/stylegan-and-stylegan2-combined-dataset/Final Dataset',
    '/content/Final_Dataset',
    regex=False
)

print(df['path'].iloc[0])
print(f"File exists : {os.path.exists(df['path'].iloc[0])}")

/content/Final_Dataset/Real/real_378_aug_1.jpg
File exists : True


In [ ]:
import os

print(os.listdir('/content/Final_Dataset'))

['Fake', 'Real', 'dataset.csv']


In [ ]:
import os

real_files = os.listdir('/content/Final_Dataset/Real')
fake_files = os.listdir('/content/Final_Dataset/Fake')

print("Real count:", len(real_files))
print("Fake count:", len(fake_files))

print("\nFirst 3 Real files:", real_files[:3])
print("First 3 Fake files:", fake_files[:3])

print("\nDF path sample:", df['path'].iloc[0])

Real count: 5890
Fake count: 7013

First 3 Real files: ['real_91_aug_1.jpg', '02613.jpg', '01348.jpg']
First 3 Fake files: ['2WK0IUKK97.jpg', 'fake_54_aug_3.jpg', 'fake_241_aug_0.jpg']

DF path sample: /content/Final_Dataset/Real/real_378_aug_1.jpg


In [ ]:
from sklearn.model_selection import train_test_split

X = df['path']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train : {len(X_train)}")
print(f"X_test  : {len(X_test)}")

X_train : 10312
X_test  : 2578


In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),          # ResNet expects 224x224
    transforms.RandomHorizontalFlip(),      # light augmentation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean
                         [0.229, 0.224, 0.225])    # ImageNet std
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torch

class CustomDataset(Dataset):

    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.transform = transform
        self.label_map = {"Fake": 0, "Real": 1}

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image    = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.label_map[self.labels[idx]]
        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
from torch.utils.data import DataLoader

train_dataset = CustomDataset(X_train, y_train, transform=train_transforms)
test_dataset  = CustomDataset(X_test,  y_test,  transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  pin_memory=True, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, pin_memory=True, num_workers=2)

print(f"Train batches : {len(train_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 323
Test batches  : 81


In [ ]:
import torch.nn as nn
from torchvision import models

def build_resnet(num_classes=1, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False


    for param in model.layer4.parameters():
        param.requires_grad = True


    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes)
    )
    return model

model = build_resnet(num_classes=1, freeze_backbone=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 75.6MB/s]


Trainable params : 8,525,825 / 11,308,609


In [ ]:
import torch.optim as optim

device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs        = 50
learning_rate = 1e-3

model     = build_resnet(num_classes=1, freeze_backbone=True).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
    weight_decay=1e-4
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

In [ ]:
from tqdm import tqdm

best_val_loss   = float('inf')
patience        = 5
patience_counter = 0

for epoch in range(epochs):


    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for features, labels in tqdm(train_loader, desc=f"[{epoch+1}/{epochs}] Train"):
        features = features.to(device, non_blocking=True)
        labels   = labels.float().to(device, non_blocking=True)

        optimizer.zero_grad()
        out  = model(features).squeeze(1)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        train_loss    += loss.item()
        preds          = (torch.sigmoid(out) >= 0.5).int()
        train_correct += (preds == labels.int()).sum().item()
        train_total   += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = train_correct / train_total * 100


    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels   = labels.float().to(device)

            out  = model(features).squeeze(1)
            loss = criterion(out, labels)

            val_loss    += loss.item()
            preds        = (torch.sigmoid(out) >= 0.5).int()
            val_correct += (preds == labels.int()).sum().item()
            val_total   += labels.size(0)

    avg_val_loss = val_loss / len(test_loader)
    val_acc      = val_correct / val_total * 100

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {avg_train_loss:.4f}  Acc: {train_acc:.2f}% | "
          f"Val Loss: {avg_val_loss:.4f}  Acc: {val_acc:.2f}%")

    # ── Early stopping + checkpoint ───────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_resnet_deepfake.pth")
        print("  ✅ Best model saved.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹ Early stopping at epoch {epoch+1}")
            break

# Load best weights after training
model.load_state_dict(torch.load("best_resnet_deepfake.pth"))

[1/50] Train: 100%|██████████| 323/323 [00:48<00:00,  6.59it/s]


Epoch 01 | Train Loss: 0.2726  Acc: 88.00% | Val Loss: 0.2064  Acc: 91.04%
  ✅ Best model saved.


[2/50] Train: 100%|██████████| 323/323 [00:48<00:00,  6.70it/s]


Epoch 02 | Train Loss: 0.1689  Acc: 93.66% | Val Loss: 0.1829  Acc: 93.87%
  ✅ Best model saved.


[3/50] Train: 100%|██████████| 323/323 [00:46<00:00,  7.01it/s]


...

[20/50] Train: 100%|██████████| 323/323 [00:47<00:00,  6.77it/s]


Epoch 20 | Train Loss: 0.0230  Acc: 99.09% | Val Loss: 0.1351  Acc: 96.16%


[21/50] Train: 100%|██████████| 323/323 [00:49<00:00,  6.57it/s]


Epoch 21 | Train Loss: 0.0093  Acc: 99.70% | Val Loss: 0.1253  Acc: 96.35%
⏹ Early stopping at epoch 21


<All keys matched successfully>

In [ ]:
import torch

def evaluate(model, loader, device):
    model.eval()

    TP = 0
    FP = 0
    FN = 0
    TN = 0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            labels   = labels.int().to(device)

            outputs = model(features).squeeze(1)

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int()

            TP += ((preds == 1) & (labels == 1)).sum().item()
            FP += ((preds == 1) & (labels == 0)).sum().item()
            FN += ((preds == 0) & (labels == 1)).sum().item()
            TN += ((preds == 0) & (labels == 0)).sum().item()

    # Metrics
    accuracy  = (TP + TN) / (TP + TN + FP + FN) * 100

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

    f1_score  = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    return accuracy, precision, recall, f1_score

In [ ]:
train_acc, train_prec, train_rec, train_f1 = evaluate(model, train_loader, device)
test_acc,  test_prec,  test_rec,  test_f1  = evaluate(model, test_loader, device)

print(f"Train Accuracy : {train_acc:.2f}%")
print(f"Train Precision: {train_prec:.4f}")
print(f"Train Recall   : {train_rec:.4f}")
print(f"Train F1 Score : {train_f1:.4f}\n")

print(f"Test Accuracy  : {test_acc:.2f}%")
print(f"Test Precision : {test_prec:.4f}")
print(f"Test Recall    : {test_rec:.4f}")
print(f"Test F1 Score  : {test_f1:.4f}")

Train Accuracy : 99.87%
Train Precision: 0.9981
Train Recall   : 0.9992
Train F1 Score : 0.9986

Test Accuracy  : 96.28%
Test Precision : 0.9632
Test Recall    : 0.9550
Test F1 Score  : 0.9591


In [ ]:
import pickle

model.eval()
model.to("cpu")

with open("deepfake_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
import torch.nn as nn
from torchvision import models

def build_resnet(num_classes=1, freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)


    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False


    for param in model.layer4.parameters():
        param.requires_grad = True


    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(256, num_classes)
    )
    return model

model = build_resnet(num_classes=1, freeze_backbone=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,}")

In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])